# 决策时锁定原价整股单：单一资产订单机械身份

固定日期/完成分钟只用当时NAV、原价收盘和现金决定SOXX或SOXL原价整数股；后续bar只尝试该资产，不把全目标权重重新投影。检查半日市、恢复前后和旧多单实例；费用、现金、原资产股数与拒单状态逐项闭合。无收益分析。

In [1]:
import os,json,hashlib,warnings
from pathlib import Path
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
warnings.filterwarnings("ignore",message="The 'generic' unit for NumPy timedelta")
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from src.core.recorder_sources import load_verified_recorder_objects
from src.backtest.stage1_event_checkpoint import capture_after_open
from src.backtest.stage1_fixed_share_event import fixed_buy_intent,settle_fixed_buy
from src.backtest.ibkr_tiered import settle_tiered_event
from src.workflow.stage1_dated_controller import DatedModel,DatedStage1Controllers
from src.workflow.stage1_f696_training_reference import prepare as prepare_f696_source
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
from copy import deepcopy
root=Path.cwd();card=root/"research/stage1_v5_fixed_raw_share_order_20261030";plan=json.loads((card/"plan.json").read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
parent=root/plan["parent_card"];old=root/plan["old_source_card"]
assert sha(parent/"summary.json")==plan["parent_summary_sha256"] and sha(parent/"verification_manifest.json")==plan["parent_manifest_sha256"]
assert sha(old/"plan.json")==plan["old_source_plan_sha256"]
assert sha(root/plan["order_audit_card"]/"audit.json")==plan["order_audit_sha256"]
assert sha(root/plan["minute_provider_uri"]/"qlib_store_manifest.json")==plan["minute_provider_manifest_sha256"]
parent_plan=json.loads((parent/"plan.json").read_text());old_plan=json.loads((old/"plan.json").read_text())
prepared=prepare_f696_source(root=root,card=old,plan=old_plan)
qlib.init(provider_uri=str(root/plan["minute_provider_uri"]),region="us",kernels=1)
names=["analysis/native/dated_f696_0","analysis/controller_audit",*[f"models/open_{y}" for y in parent_plan["signal_years"]],*[f"models/recovery_{y}" for y in (2020,2021,2022)]]
objects,receipt=load_verified_recorder_objects(parent,expected_recorder_id=plan["parent_recorder_id"],artifacts=names,expected_manifest_sha256=plan["parent_manifest_sha256"])
book=objects["analysis/native/dated_f696_0"];frame=prepared["frame"]
assert len(frame)==1056 and book["audit"]["status"]=="passed"
open_models={y:DatedModel(objects[f"models/open_{y}"],f"{y-1}-12-31",f"{parent_plan['action_recorder_id']}:open_action_value_models/{y}") for y in parent_plan["signal_years"]}
recovery_models={}
for y in parent_plan["signal_years"]:
 if y in parent_plan["explicit_no_recovery_model_years"]:recovery_models[y]=DatedModel(None,f"{y-1}-12-31",f"explicit_no_model_{y}_insufficient_prior_teacher",explicit_no_model=True)
 else:recovery_models[y]=DatedModel(objects[f"models/recovery_{y}"],f"{y-1}-12-31",f"{plan['parent_recorder_id']}:models/recovery_{y}")
controller=DatedStage1Controllers(calendar=frame[["signal_date","trade_date"]],open_models=open_models,recovery_models=recovery_models)
pd.testing.assert_frame_equal(controller.audit(),objects["analysis/controller_audit"],check_dtype=False)
recovery_spec=IntradayRecoveryDirectAllocationSpec(root=root,**prepared["action_plan"]["recovery_spec"])
action_spec=Stage1OpenActionValueSpec(root=root,**prepared["action_plan"]["action_spec"])
print('dated parent and minute source verified',len(receipt))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[31768:MainThread](2026-10-30 14:39:20,697) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[31768:MainThread](2026-10-30 14:39:20,699) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[31768:MainThread](2026-10-30 14:39:20,699) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013')}


[31768:MainThread](2026-10-30 14:39:20,799) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[31768:MainThread](2026-10-30 14:39:20,800) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[31768:MainThread](2026-10-30 14:39:20,800) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


dated parent and minute source verified 13


In [2]:
from src.backtest.event_guard_clock import guard_slot
from copy import deepcopy
by_date={};quotes_by_date={};reads=0
for item in plan['cases']:
 day=pd.Timestamp(item['trade_date'])
 if day in by_date:continue
 checkpoint=capture_after_open(frame,event_day=day,controller_by_date=controller,recovery_spec=recovery_spec,action_spec=action_spec,selection=prepared['selection'],plan=parent_plan)
 by_date[day]=checkpoint
 bars=D.features(['SOXL','SOXX'],['$open','$close','$raw_open','$raw_close','$price_valid'],start_time=str(day.date())+' 09:31:00',end_time=str(day.date())+' 16:00:00',freq='1min')
 parts=[]
 for symbol in ('SOXL','SOXX'):
  x=bars.xs(symbol,level='instrument').copy();x.columns=[symbol+'_'+k.removeprefix('$') for k in x.columns];parts.append(x)
 quotes_by_date[day]=pd.concat(parts,axis=1).sort_index();reads+=1
assert reads==plan['budget']['qlib_minute_reads']
print('source read days',reads)

source read days 4


In [3]:
results=[]
for item in plan['cases']:
 day=pd.Timestamp(item['trade_date']);checkpoint=by_date[day];quotes=quotes_by_date[day]
 bar_end=pd.Timestamp(str(day.date())+' '+item['bar_end_at']+':00');close=quotes.index.max()
 slot=guard_slot(bar_end,session_open=day+pd.Timedelta(hours=9,minutes=30),session_close=close,original_open_fill=day+pd.Timedelta(hours=9,minutes=47))
 assert slot is not None and bar_end in quotes.index and slot.fill_at+pd.Timedelta(minutes=1) in quotes.index
 late=bar_end>day+pd.Timedelta(hours=10,minutes=32)
 q=checkpoint.state.q.copy();cash=float(checkpoint.state.cash);fees=deepcopy(checkpoint.state.fees)
 recovery_orders=[]
 if late:
  pos=book['native_positions'][day+pd.Timedelta(hours=10,minutes=32)]
  q=np.array([pos.get_stock_amount(s) if pos.check_stock(s) else 0. for s in ('SOXL','SOXX')],float);cash=float(pos.get_cash())
  recovery_orders=book['orders'].loc[(book['orders'].trade_date==day)&(book['orders'].slot=='recovery')].to_dict('records')
  for order in recovery_orders:fees.commit(int(order['raw_shares']),stamp=order['stamp'],cap_applied=bool(order['cap_applied']))
 mark=quotes.loc[bar_end];fillbar=quotes.loc[slot.fill_at+pd.Timedelta(minutes=1)]
 assert mark.SOXL_price_valid==mark.SOXX_price_valid==fillbar.SOXL_price_valid==fillbar.SOXX_price_valid==1
 price=np.array([mark.SOXL_close,mark.SOXX_close],float);nav=float(q@price+cash)
 raw_close=float(mark[item['asset']+'_raw_close'])
 intent=fixed_buy_intent(symbol=item['asset'],decision_nav=nav,raw_close=raw_close,cash=cash,fees=fees,decision_at=slot.decision_at)
 assert intent['status']=='ready' and intent['raw_shares']>=1
 adjusted=np.array([fillbar.SOXL_open,fillbar.SOXX_open],float);raw=np.array([fillbar.SOXL_raw_open,fillbar.SOXX_raw_open],float)
 branch=settle_fixed_buy(q,cash,intent=intent,adjusted_fill_prices=adjusted,raw_fill_prices=raw,fill_at=slot.fill_at,fees=deepcopy(fees))
 other=1 if item['asset']=='SOXL' else 0
 assert branch['status']=='funded' and len(branch['orders'])==1 and branch['orders'][0]['symbol']==item['asset']
 assert branch['quantities'][other]==q[other] and branch['cash']>=0 and branch['orders'][0]['raw_shares']==intent['raw_shares']
 before=float(q@adjusted+cash);after=float(branch['quantities']@adjusted+branch['cash']);fee=float(branch['orders'][0]['total'])
 assert abs(after-(before-fee))<1e-7
 results.append({'case':item['reason'],'day':str(day.date()),'bar_end_at':str(bar_end),'fill_at':str(slot.fill_at),'asset':item['asset'],'parent_recovery_orders_before_decision':len(recovery_orders),'decision_raw_shares':intent['raw_shares'],'filled_raw_shares':branch['orders'][0]['raw_shares'],'orders':len(branch['orders']),'unchanged_other_asset_quantity_error':float(abs(branch['quantities'][other]-q[other])),'fee_usd':fee,'fill_nav_fee_closure_error_usd':abs(after-before+fee),'cash_after':branch['cash'],'session_close':str(close)})
# Synthetic no-fill paths use fixed quantities, never future-price resizing.
fees=deepcopy(checkpoint.state.fees)
zero=fixed_buy_intent(symbol='SOXX',decision_nav=100.,raw_close=1000.,cash=100.,fees=fees,decision_at=day+pd.Timedelta(hours=9,minutes=49))
assert zero['status']=='decision_no_whole_share' and zero['raw_shares']==0
zero_result=settle_fixed_buy(np.zeros(2),100.,intent=zero,adjusted_fill_prices=[1000.,1000.],raw_fill_prices=[1000.,1000.],fill_at=day+pd.Timedelta(hours=9,minutes=50),fees=fees)
assert zero_result['status']=='decision_no_whole_share' and not zero_result['orders']
limited=fixed_buy_intent(symbol='SOXX',decision_nav=200.,raw_close=10.,cash=200.,fees=fees,decision_at=day+pd.Timedelta(hours=9,minutes=49))
assert limited['status']=='ready'
rejected=settle_fixed_buy(np.zeros(2),200.,intent=limited,adjusted_fill_prices=[100.,100.],raw_fill_prices=[100.,100.],fill_at=day+pd.Timedelta(hours=9,minutes=50),fees=fees)
assert rejected['status']=='insufficient_fill_cash' and not rejected['orders'] and rejected['cash']==200.
audit={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'order_code_sha256':sha(root/'src/backtest/stage1_fixed_share_event.py'),'cases':results,'maximum_other_asset_quantity_error':max(x['unchanged_other_asset_quantity_error'] for x in results),'maximum_fill_nav_fee_error_usd':max(x['fill_nav_fee_closure_error_usd'] for x in results),'synthetic_decision_zero_share_retained':True,'synthetic_fill_cash_rejection_retained':True,'qlib_minute_reads':reads,'new_model_fits':0,'new_candidate_accounts':0,'historical_research_passed':False,'decision':'causal_fixed_raw_share_one_asset_order_mechanical_pass'}
(card/'audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2)+'\n')
print(json.dumps(audit,ensure_ascii=False,indent=2))

{
  "experiment": "stage1_v5_fixed_raw_share_order_20261030",
  "plan_sha256": "524c87afdbd0e183f6449e875cce402e27fab08eb40831b864ce7ac2123b1458",
  "order_code_sha256": "5bf69f1d17e266d869ce13ba6511ea8b1675b1b24e3d382fe4285f20d3deefe7",
  "cases": [
    {
      "case": "first source-only old extra SOXL sale",
      "day": "2018-10-15",
      "bar_end_at": "2018-10-15 11:25:00",
      "fill_at": "2018-10-15 11:27:00",
      "asset": "SOXX",
      "parent_recovery_orders_before_decision": 0,
      "decision_raw_shares": 59,
      "filled_raw_shares": 59,
      "orders": 1,
      "unchanged_other_asset_quantity_error": 0.0,
      "fee_usd": 0.5392342499999999,
      "fill_nav_fee_closure_error_usd": 1.883049272066728e-12,
      "cash_after": 52148.14228943418,
      "session_close": "2018-10-15 16:00:00"
    },
    {
      "case": "13:00 half-day",
      "day": "2018-11-23",
      "bar_end_at": "2018-11-23 12:00:00",
      "fill_at": "2018-11-23 12:02:00",
      "asset": "SOXX",
      "p

## 中文结论

SOXX历史门未运行。固定决策bar原价收盘、NAV和现金先确定整数股数，下一bar只尝试该资产原价整股买单；五个事前时钟均恰一笔预定买单，原持仓另一资产股数差0，填单NAV=填单前NAV−Tiered费最大误差3.01e−11美元。旧2018-10-15 11:25目标权重版会卖1股SOXL再买59股SOXX；新固定股数版只买59股SOXX。2018半日市13:00、2019普通早盘、2020恢复前SOXL和恢复后SOXX均闭合。合成不足一股及填单现金不足分别返回decision_no_whole_share/insufficient_fill_cash，绝不按未来填单价缩减预定股数。此卡只是订单机械身份，不能挪用旧五会话标签或称找到了超额；下一步将其接入同状态锁定回放并另生标签。